In [27]:
import warnings
warnings.filterwarnings("ignore")

In [28]:
import sys
sys.path.append(r'C:\Users\julia\OneDrive\Escritorio\Trabajo\building_ml_models_for_protein_science\src')

In [29]:
from building_models.commons_functions.parsers_commons import ParsersCommons
from building_models.utils.constants import COLUMNS_TO_WORK
from building_models.utils.utils_functions import UtilsFunctions
import pandas as pd

- Read doc and labels

In [30]:
path_export = "../../processed_dataset/"
path_input = "../../raw_dataset/"
metadata_file = "../../raw_dataset/raw_data_description.xlsx"
name_task = "antioxidant_classification"
name_source = "Feng et al"

In [31]:
import re
import pandas as pd

# Leer el texto extraído del PDF
with open(f"{path_input}/{name_source}/full_text.txt", 'r') as f:
    content = f.read()

content = content.replace('\f', '')  # limpiar saltos de página

records = []
current_id = None
current_seq = []

for line in content.split('\n'):
    line = line.strip()
    if line.startswith('>'):
        if current_id:
            records.append({'id': current_id, 'sequence': ''.join(current_seq)})
        current_id = line[1:]
        current_seq = []
    elif current_id and re.match(r'^[A-Z]+$', line):
        current_seq.append(line)

if current_id:
    records.append({'id': current_id, 'sequence': ''.join(current_seq)})

df_data = pd.DataFrame(records)
df_data['label'] = df_data['id'].apply(lambda x: 1 if x.startswith('antioxidant') or len(x)==6 else 0)

In [32]:
df_data["label"] = df_data["label"].astype(int)
df_data = df_data.drop(columns=["id"])
df_data

,sequence,label
0,MTKGILLGDKFPDFRAETNEGFIPSFYDWIGKDSWAILFSHPRDFT...,1
1,MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...,1
2,MAIALSSSSTITSITLQPKLKTIHGLGTVLPGYSVKSHFRSVSLRR...,1
3,MITSSKKIVSAMLSTSLWIGVASAAYAETTNVEAEGYSTIGGTYQD...,1
4,MANSGLWELITIGSAVRNVAKSYLKAEASSITAKQLYDASKITSSK...,1
...,...,...
1836,MVKAVAVLGSSDGVKGTIFFTQEGDGPTAVTGSVSGLKPGLHGFHV...,1
1837,MASHSLMSPSPLTSHSLLRSSFSGVSVKLSPQFSTLSRSKFQPLSV...,1
1838,MQAILAAAMAAQTLLFSATAPPASLFQSPSSARPFHSLRLAAGPAG...,1
1839,MASQTLVSPSPLSSHSLLRTSFSGVSVKLAPQFSTLATSNFKPLTV...,1


- Checking duplicates

In [33]:
df_consistent_duplicates, df_errors, df_unique = ParsersCommons.processing_duplicated(
    df_data, group_seq= "sequence",
    label_col= "label")
df_consistent_duplicates.shape, df_errors.shape, df_unique.shape
#no duplicates

((1, 3), (0, 0), (1839, 2))

In [34]:
data_correct = pd.concat([df_consistent_duplicates, df_unique], axis=0, ignore_index=True)
data_correct = data_correct.drop(columns=["n_duplicates"])
data_correct.head()

,sequence,label
0,DNDSVVEEHGQLSISNGELVNERGEQVQLKGMSSHGLQWYGQFVNY...,0
1,MTKGILLGDKFPDFRAETNEGFIPSFYDWIGKDSWAILFSHPRDFT...,1
2,MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...,1
3,MAIALSSSSTITSITLQPKLKTIHGLGTVLPGYSVKSHFRSVSLRR...,1
4,MITSSKKIVSAMLSTSLWIGVASAAYAETTNVEAEGYSTIGGTYQD...,1


- Reading metadata

In [35]:
metadata_file = ParsersCommons.read_metadata(metadata_file, name_source=name_source, columns_to_select=COLUMNS_TO_WORK)
metadata_file.head()

,name dataset,name source,type source,static-dynamic,license,reports constant updates,year of publication,last update date,download date,file format,protein format,category dataset,task,obtaining negative dataset,obtaining positive dataset,repository or server,publication,unit of measurement
45,567529.f1.pdf,Feng et al,Dataset,Static,NaN,No,2013,NaT,2026-04-07,docx,Sequence,NaN,Antioxidant,NaN,NaN,Supplementary Material,https://pmc.ncbi.nlm.nih.gov/articles/PMC3766563/,NaN


In [36]:
dict_metadata = ParsersCommons.create_metadata_from_file(metadata_file)
dict_metadata

{'name dataset': '567529.f1.pdf',
 'name source': 'Feng et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': nan,
 'reports constant updates': 'No',
 'year of publication': 2013,
 'last update date': NaT,
 'download date': Timestamp('2026-04-07 00:00:00'),
 'file format': 'docx',
 'protein format': 'Sequence',
 'category dataset': nan,
 'task': 'Antioxidant',
 'obtaining negative dataset': nan,
 'obtaining positive dataset': nan,
 'repository or server': 'Supplementary Material',
 'publication': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3766563/',
 'unit of measurement': nan,
 'number_of_sources': 1,
 'processing_date': '2026-04-08 16:15:20'}

In [37]:
dict_metadata['number_of_records'] = df_data.shape[0]
dict_metadata['number_of_collected_sequences'] = df_data.shape[0]
dict_metadata['number_of_unique_sequences'] = data_correct.shape[0]
dict_metadata['positive_examples'] = data_correct[data_correct["label"] == 1].shape[0]
dict_metadata['negative_examples'] = data_correct[data_correct["label"] == 0].shape[0]
dict_metadata['number_of_sequences_with_errors'] = df_errors.shape[0]
dict_metadata

{'name dataset': '567529.f1.pdf',
 'name source': 'Feng et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': nan,
 'reports constant updates': 'No',
 'year of publication': 2013,
 'last update date': NaT,
 'download date': Timestamp('2026-04-07 00:00:00'),
 'file format': 'docx',
 'protein format': 'Sequence',
 'category dataset': nan,
 'task': 'Antioxidant',
 'obtaining negative dataset': nan,
 'obtaining positive dataset': nan,
 'repository or server': 'Supplementary Material',
 'publication': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3766563/',
 'unit of measurement': nan,
 'number_of_sources': 1,
 'processing_date': '2026-04-08 16:15:20',
 'number_of_records': 1841,
 'number_of_collected_sequences': 1841,
 'number_of_unique_sequences': 1840,
 'positive_examples': 274,
 'negative_examples': 1566,
 'number_of_sequences_with_errors': 0}

- Export data

In [39]:
UtilsFunctions.make_directory(f"{path_export}{name_task}/{name_source}")
UtilsFunctions.export_json(f"{path_export}{name_task}/{name_source}/metadata_{name_source}.json", dict_metadata)
data_correct.to_csv(f"{path_export}{name_task}/{name_source}/processed_data.csv", index=False)